# Module 03: Nlp Pipeline


# 3.1 Pipeline Overview


## 🚂 The spaCy Processing Pipeline

When you call `nlp(text)`, spaCy doesn't just do one thing. It passes the text through a series of processing steps called a **pipeline**.

The standard flow looks like this:
`Text` ➡️ **Tokenizer** ➡️ `Doc` ➡️ **Tagger** ➡️ **Parser** ➡️ **NER** ➡️ ... ➡️ `Processed Doc`

- The **Tokenizer** is special: it's the *only* component that takes raw text and turns it into a `Doc`.
- All other components take the `Doc`, add annotations to it (like POS tags or entities), and pass it to the next component.


In [1]:
import spacy
import pandas as pd

# Load a standard pipeline
nlp = spacy.load("en_core_web_sm")

# Let's see what components are in this pipeline and in what order!
pipeline_data = []
for name, component in nlp.pipeline:
    pipeline_data.append({
        "Component Name": name,
        "Component Type": type(component).__name__
    })

pd.DataFrame(pipeline_data)


,Component Name,Component Type
0,tok2vec,Tok2Vec
1,tagger,Tagger
2,parser,DependencyParser
3,attribute_ruler,AttributeRuler
4,lemmatizer,EnglishLemmatizer
5,ner,EntityRecognizer


## 🌊 Data Flow

Because the components run sequentially, the order matters.
- The **Lemmatizer** needs Part-of-Speech tags to accurately find the base word, so it must run *after* the **Tagger**.
- The **Dependency Parser** uses POS tags as features, so it also usually runs *after* the **Tagger**.

Let's see the total data flow of the `nlp` object:


In [2]:
# The nlp object exposes its pipeline names directly as a list
print("Pipeline execution order:")
print("Tokenizer -> " + " -> ".join(nlp.pipe_names))


Pipeline execution order:
Tokenizer -> tok2vec -> tagger -> parser -> attribute_ruler -> lemmatizer -> ner



<br><br>

---

<br><br>


# 3.2 Built-in Components


## 🧱 Core Pipeline Components

Let's break down exactly what the main built-in components do.

### 1. Tokenizer (Not strictly a pipeline component)
- **Input**: Raw text (`str`)
- **Output**: `Doc` object
- **Function**: Splits text into words, punctuation, etc. based on language-specific rules.

### 2. tok2vec (Token-to-Vector)
- **Function**: Calculates context-sensitive vectors (embeddings) for each token. Other components (like the tagger and parser) use these vectors as input to make their predictions. It's an efficiency optimization.

### 3. tagger (Part-of-Speech Tagger)
- **Function**: Assigns basic POS tags (e.g., NOUN, VERB) and detailed tags (e.g., VBD for verb past tense).


In [1]:
import spacy
nlp = spacy.load("en_core_web_sm")
doc = nlp("I am learning about spaCy pipelines.")

print("--- TAGGER OUTPUT ---")
for token in doc:
    print(f"{token.text:<10} | POS: {token.pos_:<5} | Tag: {token.tag_}")


--- TAGGER OUTPUT ---
I          | POS: PRON  | Tag: PRP
am         | POS: AUX   | Tag: VBP
learning   | POS: VERB  | Tag: VBG
about      | POS: ADP   | Tag: IN
spaCy      | POS: NUM   | Tag: CD
pipelines  | POS: NOUN  | Tag: NNS
.          | POS: PUNCT | Tag: .


### 4. parser (Dependency Parser)
- **Function**: Predicts syntactic dependencies (how words relate to each other). It also detects sentence boundaries (`doc.sents`).


In [2]:
print("--- PARSER OUTPUT ---")
for token in doc:
    print(f"{token.text:<10} | Dependency: {token.dep_:<10} | Head: {token.head.text}")


--- PARSER OUTPUT ---
I          | Dependency: nsubj      | Head: learning
am         | Dependency: aux        | Head: learning
learning   | Dependency: ROOT       | Head: learning
about      | Dependency: prep       | Head: learning
spaCy      | Dependency: nummod     | Head: pipelines
pipelines  | Dependency: pobj       | Head: about
.          | Dependency: punct      | Head: learning


### 5. ner (Named Entity Recognizer)
- **Function**: Identifies and labels named entities (e.g., PERSON, ORG, GPE).


In [3]:
doc2 = nlp("Apple is looking at buying U.K. startup for $1 billion")
print("--- NER OUTPUT ---")
for ent in doc2.ents:
    print(f"{ent.text:<15} | Label: {ent.label_}")


--- NER OUTPUT ---
Apple           | Label: ORG
U.K.            | Label: GPE
$1 billion      | Label: MONEY


### 6. lemmatizer
- **Function**: Assigns the base form of the word (e.g., "am" -> "be", "learning" -> "learn").


In [4]:
print("--- LEMMATIZER OUTPUT ---")
for token in doc:
    print(f"Word: {token.text:<10} | Lemma: {token.lemma_}")


--- LEMMATIZER OUTPUT ---
Word: I          | Lemma: I
Word: am         | Lemma: be
Word: learning   | Lemma: learn
Word: about      | Lemma: about
Word: spaCy      | Lemma: spacy
Word: pipelines  | Lemma: pipeline
Word: .          | Lemma: .



<br><br>

---

<br><br>


# 3.3 Pipeline Management


## ⚙️ Managing the Pipeline

You don't have to accept the default pipeline. You can add, remove, disable, or replace components to suit your needs.

### Disabling Components for Speed
If you only need Tokenization and NER, running the Tagger and Parser is a waste of time and computational power.
You can disable components when loading the model:


In [1]:
import spacy

# Load the model but disable the tagger, parser, and lemmatizer
# This makes processing much faster!
nlp_fast = spacy.load("en_core_web_sm", disable=["tagger", "parser", "lemmatizer"])

print("Active components:", nlp_fast.pipe_names)


Active components: ['tok2vec', 'attribute_ruler', 'ner']


### Adding Components
You can add built-in or custom components to the pipeline using `nlp.add_pipe()`. You can specify exactly where it should go using `before`, `after`, `first`, or `last`.


In [2]:
nlp = spacy.load("en_core_web_sm")

# Let's add the built-in 'sentencizer' component
# It's a faster, rule-based way to split sentences than the dependency parser
if "sentencizer" not in nlp.pipe_names:
    nlp.add_pipe("sentencizer", before="parser")

print("Updated pipeline:", nlp.pipe_names)


Updated pipeline: ['tok2vec', 'tagger', 'sentencizer', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']


### Removing and Replacing Components
You can easily strip components out permanently.


In [3]:
# Remove the NER component
if "ner" in nlp.pipe_names:
    nlp.remove_pipe("ner")

print("Pipeline without NER:", nlp.pipe_names)


Pipeline without NER: ['tok2vec', 'tagger', 'sentencizer', 'parser', 'attribute_ruler', 'lemmatizer']



<br><br>

---

<br><br>


# 3.4 Efficient Text Processing


## 🏎️ Batch Processing with `nlp.pipe()`

Calling `doc = nlp(text)` is great for a single string. But if you have 10,000 tweets, doing it in a `for` loop is inefficient.

```python
# SLOW:
docs = [nlp(text) for text in lots_of_texts]
```

Instead, use `nlp.pipe()`. It processes texts as a stream (generator) and yields `Doc` objects. It batches texts together internally, which is much faster—especially if you are using a GPU or running multiple processes.


In [1]:
import spacy
import time

nlp = spacy.load("en_core_web_sm")

texts = [
    "This is text 1.",
    "This is the second text.",
    "Here is yet another text to process.",
    "Batch processing is much faster!"
] * 1000  # 4,000 texts total

# Method 1: For loop (Slow)
start = time.time()
docs_loop = [nlp(text) for text in texts]
print(f"For loop took: {time.time() - start:.2f} seconds")

# Method 2: nlp.pipe (Fast)
start = time.time()
docs_pipe = list(nlp.pipe(texts, batch_size=50))
print(f"nlp.pipe took: {time.time() - start:.2f} seconds")


For loop took: 43.18 seconds
nlp.pipe took: 16.33 seconds


## 🔄 Processing with Context (Tuples)

Often, you have metadata alongside your text (like a tweet ID or author name). You want to keep that metadata attached to the processed `Doc`.

You can pass tuples of `(text, context)` to `nlp.pipe` by setting `as_tuples=True`.


In [2]:
data = [
    ("I love this product!", {"id": 1, "author": "Alice"}),
    ("Terrible customer service.", {"id": 2, "author": "Bob"}),
]

# Pass the data, yielding (Doc, context) pairs
for doc, context in nlp.pipe(data, as_tuples=True):
    print(f"Author: {context['author']} | Entities found: {len(doc.ents)}")


Author: Alice | Entities found: 0
Author: Bob | Entities found: 0


## 💻 Multiprocessing

You can easily use multiple CPU cores by setting the `n_process` argument in `nlp.pipe()`. Note: this requires scripts to be run in a proper `if __name__ == '__main__':` block on Windows, but works seamlessly in Jupyter/Linux.


In [ ]:
# Example of using 2 CPU cores
# docs = list(nlp.pipe(texts, n_process=2))



<br><br>

---

<br><br>


# 3.5 Debugging the Pipeline


## 🐛 Debugging spaCy Pipelines

Sometimes things don't work as expected. spaCy has built-in tools to help you figure out what's wrong with your pipeline configuration.

### Inspecting Pipeline Information
You can easily check what a component requires as input, and what it produces as output.


In [1]:
import spacy

nlp = spacy.load("en_core_web_sm")

# Analyze a specific component, like the 'tagger'
print("Tagger Analysis:")
analysis = nlp.analyze_pipes()

# Let's format the output nicely
import pandas as pd
pd.DataFrame(analysis['summary'])


Tagger Analysis:


,tok2vec,tagger,parser,attribute_ruler,lemmatizer,ner
assigns,[doc.tensor],[token.tag],"[token.dep, token.head, token.is_sent_start, d...",[],[token.lemma],"[doc.ents, token.ent_iob, token.ent_type]"
requires,[],[],[],[],[],[]
scores,[],"[tag_acc, pos_acc, tag_micro_p, tag_micro_r, t...","[dep_uas, dep_las, dep_las_per_type, sents_p, ...",[],[lemma_acc],"[ents_f, ents_p, ents_r, ents_per_type]"
retokenizes,False,False,False,False,False,False


### Profiling Performance
If your pipeline is running slowly, you can profile it to see which component is taking the most time. A simple way to do this is to iterate through `nlp.pipeline` and use Python's `time` module.


In [2]:
import spacy
import time

nlp = spacy.load("en_core_web_sm")
text = "This is a sentence to profile. " * 100

# Create an empty doc
doc = nlp.make_doc(text)

print("Profiling complete! Time spent in each component:")
for name, component in nlp.pipeline:
    start_time = time.perf_counter()
    # Apply the component to the doc
    doc = component(doc)
    end_time = time.perf_counter()
    print(f"- {name:<10}: {end_time - start_time:.5f} seconds")


Profiling complete! Time spent in each component:
- tok2vec   : 0.09960 seconds
- tagger    : 0.00179 seconds
- parser    : 0.06781 seconds
- attribute_ruler: 0.04717 seconds
- lemmatizer: 0.00497 seconds
- ner       : 0.15151 seconds


## 🎉 Summary of Part 1 (Foundations)

Congratulations! You have completed **Part 1: Foundations**.
You now understand:
1. How to install and set up spaCy.
2. How the `Doc`, `Token`, `Span`, and `Vocab` memory systems work.
3. How to manage, reorder, and optimize the NLP processing pipeline using `nlp.pipe()`.

In **Part 2**, we will dive into **Core NLP Features**, starting with Module 4: Tokenization & Text Processing.
